In [1]:

# 1. IMPORTS

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = "/content/drive/MyDrive/full_dataset.csv"

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

# Load the dataset
data = pd.read_csv('/content/drive/MyDrive/full_dataset.csv')

# Print the first few rows of the dataset to check its contents
print(data.head())

    label  acc_x_0  acc_x_1  acc_x_2  acc_x_3  acc_x_4  acc_x_5  acc_x_6  \
0    idle   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   
1    fall  -0.0008   0.0181   0.1298   0.1815   0.1867   0.0526   0.0188   
2    step   0.0200   0.0200   0.0276   0.0201   0.0000   0.0000   0.0000   
3  motion  -0.0348  -0.0104   0.0309   0.0373   0.0498   0.0600   0.0984   
4    step   0.0200   0.0172   0.0100   0.0020  -0.0099   0.0024   0.0155   

   acc_x_7  acc_x_8  ...  gy_z_150  gy_z_151  gy_z_152  gy_z_153  gy_z_154  \
0   0.0000   0.0000  ...    1.6440    1.6856    1.5975    1.4654    1.5300   
1   0.0228   0.3152  ...    1.2886    1.2177    1.3184    1.1846    0.7587   
2   0.0107   0.0291  ...   -0.5565   -0.1801   -0.2183   -0.3190   -0.3448   
3   0.0810   0.0559  ...   -0.5500   -0.5500   -0.2313   -0.1530   -0.1978   
4   0.0357   0.0200  ...   -0.5644   -0.5155   -0.5806   -0.6057   -0.5801   

   gy_z_155  gy_z_156  gy_z_157  gy_z_158  gy_z_159  
0    1.5300    1.590

In [4]:
# Features & Labels
X = data.iloc[:, 1:].values   # 960 features
y = data.iloc[:, 0].values    # labels

In [5]:

# 3. ENCODE LABELS

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(label, "=", i)


Label Mapping:
fall = 0
idle = 1
motion = 2
step = 3


In [17]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print(class_weights_dict)

{0: np.float64(1.0184804928131417), 1: np.float64(0.9880478087649402), 2: np.float64(0.9939879759519038), 3: np.float64(1.0)}


In [18]:

# 4. NORMALIZE DATA

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# SAVE scaler values ( IMPORTANT for Android)
np.save('/content/drive/MyDrive/scaler_min.npy', scaler.data_min_)
np.save('/content/drive/MyDrive/scaler_max.npy', scaler.data_max_)

print("✅ Scaler saved")

✅ Scaler saved


In [23]:
# reshape into (samples, timesteps, features)
X_reshaped = X_scaled.reshape(-1, 160, 6)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_reshaped, y_encoded, test_size=0.2, random_state=42
)

In [24]:

# 6. BUILD FINAL MODEL

model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(64, 3, activation='relu', input_shape=(160, 6)),
    tf.keras.layers.MaxPooling1D(2),

    tf.keras.layers.Conv1D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:

# 7. TRAIN MODEL (SMART TRAINING)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1
)



Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 8s 73ms/step - accuracy: 0.2880 - loss: 1.3889 - val_accuracy: 0.5528 - val_loss: 1.3015
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5373 - loss: 1.1066 - val_accuracy: 0.5779 - val_loss: 0.9498
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6095 - loss: 0.8713 - val_accuracy: 0.6181 - val_loss: 0.7908
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6291 - loss: 0.7783 - val_accuracy: 0.6382 - val_loss: 0.7386
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6510 - loss: 0.7161 - val_accuracy: 0.6784 - val_loss: 0.5986
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6801 - loss: 0.6634 - val_accuracy: 0.6935 - val_loss: 0.5763
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6717 - loss: 0.6678 - val_accuracy: 0.6834 - val_loss: 0.5798
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6908 - loss: 0.6351 - val_accuracy: 0.6834 - val_loss

In [26]:

# 8. EVALUATE

loss, acc = model.evaluate(X_test, y_test)
print("✅ Test Accuracy:", acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9032 - loss: 0.3238
✅ Test Accuracy: 0.9032257795333862


In [27]:

# 9. CONVERT TO TFLITE

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

Saved artifact at '/tmp/tmp5dthqw_e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 6), dtype=tf.float32, name='keras_tensor_37')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  138858109609232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109611536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109613840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109608656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109608848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109610576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109611728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138858109610768: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [28]:
# SAVE TFLITE MODEL
with open('/content/drive/MyDrive/fall_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ TFLite model saved successfully!")

✅ TFLite model saved successfully!
